# Day 44: Optimisation – vLLM for High‑Performance Serving

vLLM dramatically improves LLM inference throughput (up to 24x higher than naive Hugging Face).

In [ ]:
# 1. Start a vLLM server (run this in terminal, not notebook)
vllm_command = """
python -m vllm.entrypoints.api_server \
    --model meta-llama/Llama-2-7b-chat-hf \
    --tensor-parallel-size 1 \
    --max-num-batched-tokens 4096
"""
print("Run this command in a terminal after installing vLLM:")
print(vllm_command)

In [ ]:
# 2. Send requests to vLLM server (once running)
import requests
import json

url = "http://localhost:8000/generate"
prompt = "Explain quantum computing in one sentence."

payload = {
    "prompt": prompt,
    "max_tokens": 100,
    "temperature": 0.7,
    "top_p": 0.95,
}

response = requests.post(url, json=payload)
if response.status_code == 200:
    print(response.json()["text"])
else:
    print(f"Error: {response.status_code}")

## 2. Using vLLM as a Python library (offline, no server)

In [ ]:
from vllm import LLM, SamplingParams

# Load model (this will take some time and GPU memory)
llm = LLM(model="meta-llama/Llama-2-7b-chat-hf", tensor_parallel_size=1)

sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=100)

prompts = [
    "What is the capital of France?",
    "Write a haiku about AI.",
]
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt}\nGenerated: {generated_text}\n")

## 3. Benchmarking vs. Hugging Face Pipeline

In [ ]:
import time

# HF pipeline (slower)
from transformers import pipeline
hf_pipe = pipeline("text-generation", model="meta-llama/Llama-2-7b-chat-hf", device=0)

start = time.time()
for prompt in prompts:
    hf_pipe(prompt, max_new_tokens=100)
hf_time = time.time() - start

# vLLM offline
start = time.time()
outputs = llm.generate(prompts, sampling_params)
vllm_time = time.time() - start

print(f"HF pipeline time: {hf_time:.2f}s")
print(f"vLLM time: {vllm_time:.2f}s")
print(f"Speedup: {hf_time/vllm_time:.2f}x")

## 4. Text Generation Inference (TGI) alternative
Run TGI via Docker:
```
docker run --gpus all -p 8080:80 ghcr.io/huggingface/text-generation-inference:latest --model-id meta-llama/Llama-2-7b-chat-hf
```
Then query:
```python
import requests
response = requests.post('http://localhost:8080/generate', json={'inputs': prompt})
```